# Production Feedback and Drift: Turn Incidents into Better Tests

> **The story:** In 1950, W. Edwards Deming taught that quality improves through a measured feedback loop rather than final inspection alone. Modern ML monitoring inherits that discipline but adds a dangerous input: production interactions can contain personal, confidential, or regulated content. Riverside needs the loop without turning observability into a second data leak.
>
> **Where you are:** Riverside can retrieve approved policy, evaluate retrieval and generation separately, route requests through a gateway, and attribute latency and cost. Release `rel-riv-002` is now serving synthetic production traffic. The missing capability is deciding whether changed behavior belongs to traffic, data, retrieval, generation quality, operations, cost, or policy, then feeding only reviewed evidence back into evaluation.
>
> **Notation:** $B$ is the baseline window; $C$ is the current window; $p_B$ and $p_C$ are proportions; $Delta_{pp}=100(p_C-p_B)$ is percentage-point change; $n$ is the window sample size; $z=1.96$ is the 95% Wilson interval constant.

> **Evidence banner:** `FIXTURE` inputs, `VALIDATED` deterministic outcomes, `OUTPUTS CLEARED`, `UNVALIDATED` production behavior.

This notebook uses 12 synthetic traces containing privacy-safe query categories rather than raw prompts. It makes no model, provider, network, or cloud call. Its local fixture workflow executed successfully in the unified FDE environment, then notebook outputs and execution counts were cleared.

| Step | Failure | Evidence you build |
|---|---|---|
| 0 | Feedback exists but does not control the next release | One end-to-end failure chain |
| 1 | Raw capture and naive random samples are unsafe or blind | Data-minimized, must-keep sampling |
| 2 | Malformed windows make drift arithmetic meaningless | Schema and semantic health checks |
| 3 | One quality score has no component owner | Seven separate drift lenses |
| 4 | Six rows can look more certain than they are | Wilson intervals and review zones |
| 5 | A failure list does not reveal recurring patterns | Multi-label clusters and review |
| 6 | Reviewed traces lose lineage in a spreadsheet | Versioned evaluation candidate |
| 7 | Teams change the component they know best | Five-way intervention matrix |
| 8 | A report does not close the loop | Follow-up gates and health checks |
| 9 | Fixture arithmetic becomes an overclaim | Coverage ledger and limitations |

## 0 - The Challenge

> **The mission:** Riverside House must explain why `rel-riv-002` regressed, retain no raw prompt text, create a reviewed regression candidate, and choose the smallest justified intervention.

**What we know so far:**

- `rel-riv-001` is the baseline and `rel-riv-002` is the current release.
- The fixture pins six traces per window and a latency SLO of at most 140 ms.
- Retrieval, quality, policy, latency, and cost are recorded separately.
- **But Riverside cannot yet turn those records into a governed improvement decision.**

**What's blocking us:** Quality fell from 83.3% to 50.0%, but that number cannot tell whether traffic changed, inputs are novel, retrieval missed evidence, generation ignored context, policy failed, or the route became slower and more expensive. Acting on the aggregate invites the expensive habit: fine-tune first, diagnose later.

**What this chapter unlocks:** A deterministic feedback loop that keeps content out of telemetry, reports seven drift lenses with populations, preserves multi-label failures, creates three versioned cases, and chooses retrieval/index plus guardrail work before model adaptation.

```mermaid
flowchart LR
    A["Production traces"] --> B["Failure: raw content or blind sample"]
    B --> C["Privacy-safe stratified sample"]
    C --> D["Failure: one aggregate score"]
    D --> E["Separate drift + reviewed cases"]
    E --> F["Versioned eval candidate"]
    F --> G["Smallest justified intervention"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Failure chain:** raw capture -> blind sample -> undifferentiated drift -> unreviewed failures -> unversioned cases -> habitual intervention -> missing follow-up evidence.

In [ ]:
# -- Resolve the repository and load immutable fixture bytes ------------------
from collections import Counter, defaultdict
from hashlib import sha256
import json
import math
from pathlib import Path
import random


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning/role-based-tracks/ai-engineer/shared").is_dir():
            return candidate
    searched = " -> ".join(str(candidate) for candidate in (start, *start.parents))
    raise FileNotFoundError(
        f"Could not locate the ai-portfolio repository root from {start}. "
        "Expected both AUTHORING_GUIDE.md and learning/role-based-tracks/ai-engineer/shared in one ancestor. "
        f"Searched: {searched}"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CHAPTER_DIR = REPO_ROOT / "learning/role-based-tracks/ai-engineer/05-production-feedback-and-drift"
SHARED_ROOT = REPO_ROOT / "learning/role-based-tracks/ai-engineer/shared"
SHARED_DIR = SHARED_ROOT / "feedback-drift"
TRACE_PATH = SHARED_DIR / "production-feedback.jsonl"
SCHEMA_PATH = SHARED_DIR / "production-feedback.schema.json"
fixture_version = (SHARED_ROOT / "VERSION").read_text(encoding="utf-8").strip()
fixture_manifest = json.loads((SHARED_ROOT / "fixture-manifest.json").read_text(encoding="utf-8"))
if fixture_manifest["fixture_version"] != fixture_version:
    raise RuntimeError("Fixture VERSION and fixture-manifest.json disagree")
for relative_path in (
    "feedback-drift/production-feedback.jsonl",
    "feedback-drift/production-feedback.schema.json",
    "feedback-drift/EXPECTED_OUTCOMES.md",
):
    expected_digest = fixture_manifest["files"].get(relative_path)
    if expected_digest is None:
        raise RuntimeError(f"Fixture manifest does not pin {relative_path}")
    actual_digest = sha256((SHARED_ROOT / relative_path).read_bytes()).hexdigest()
    if actual_digest != expected_digest:
        raise RuntimeError(
            f"Stale or modified fixture: {relative_path}. "
            "Restore the pinned bytes or intentionally version the shared fixture contract."
        )
trace_bytes = TRACE_PATH.read_bytes()
traces = [json.loads(line) for line in trace_bytes.decode("utf-8").splitlines() if line]
trace_schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
fixture_sha256 = sha256(trace_bytes).hexdigest()

print(f"Verified fixture contract: {fixture_version}")
print(f"Loaded {len(traces)} synthetic, privacy-safe traces.")
print(f"Fixture SHA-256: {fixture_sha256}")
print("No raw content, identity, model, provider, network, or cloud call is used.")

## 1 - Privacy-Safe Sampling Before Analysis

A feedback loop does not justify retaining everything. Operational monitoring needs categories, outcomes, timing, cost, release lineage, and bounded failure codes. Content review is a separate purpose with separate access, retention, deletion, and approval. Hashing raw text is not anonymization: repeated or guessable prompts can still be recognized.

```mermaid
flowchart TD
    A["Live request"] --> B["Data-minimization projection"]
    B --> C{"Critical or failed?"}
    C -->|yes| D["Keep privacy-safe metadata at 100%"]
    C -->|no| E["Deterministic baseline sample"]
    D --> F["Restricted review queue"]
    E --> G["Aggregate drift windows"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If Riverside keeps only two random current traces, which event is easiest to lose: a common retrieval miss, a common latency breach, or the one policy false allow?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Log prompts and completions into every span | Monitoring becomes a broad content store |
| Right | Allowlist bounded fields; approve content review separately | Purpose, access, and retention stay explicit |
| Wrong | Sample uniformly at trace start | Rare failures and minority slices disappear |
| Right | Keep critical outcomes and stratify routine traffic | Safety and population coverage survive |

**Quick Health Check:** forbidden content fields are absent, every critical event is kept, and routine inclusion is stable for the same trace ID.

**Your turn:** Change the routine baseline sampling rate in the next cell. Predict how many ordinary rows move, while the policy false allow must remain retained.

In [ ]:
# -- Prove that must-keep strata protect rare policy failures -----------------
current_rows = [row for row in traces if row["window"] == "current"]
naive_sample = random.Random(2026).sample(current_rows, k=2)
naive_ids = {row["feedback_trace_id"] for row in naive_sample}
policy_false_allow_ids = {
    row["feedback_trace_id"]
    for row in current_rows
    if "policy_false_allow" in row["failure_codes"]
}


def deterministic_baseline_keep(trace_id: str, rate_percent: int = 20) -> bool:
    bucket = int(sha256(trace_id.encode("utf-8")).hexdigest()[:8], 16) % 100
    return bucket < rate_percent


def sampling_reason(row: dict, rate_percent: int = 20) -> str:
    if "policy_false_allow" in row["failure_codes"]:
        return "must_keep_policy"
    if row["failure_codes"] or row["user_feedback"] == "down":
        return "must_keep_failure"
    return (
        "baseline_sample"
        if deterministic_baseline_keep(row["feedback_trace_id"], rate_percent)
        else "aggregate_only"
    )


sampling_plan = {row["feedback_trace_id"]: sampling_reason(row) for row in current_rows}
forbidden_fields = {
    "prompt", "raw_prompt", "completion", "response_text", "manuscript_text",
    "user_id", "tenant_id", "email", "authorization",
}
present_fields = set().union(*(row.keys() for row in traces))
sampling_health = {
    "forbidden_fields_absent": not forbidden_fields.intersection(present_fields),
    "policy_events_kept": all(
        sampling_plan[trace_id] == "must_keep_policy" for trace_id in policy_false_allow_ids
    ),
    "stable_routine_decision": all(
        deterministic_baseline_keep(row["feedback_trace_id"])
        == deterministic_baseline_keep(row["feedback_trace_id"])
        for row in current_rows
    ),
}
print(f"Naive sample: {sorted(naive_ids)}; kept false allow: {policy_false_allow_ids <= naive_ids}")
print("Prediction resolved: the rare false allow is easiest to lose under a two-row random sample.")
print("Outcome-aware plan:", sampling_plan)
for name, passed in sampling_health.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(sampling_health.values())

# Your turn: change routine sampling, never critical retention.
BASELINE_RATE_PERCENT = 40  # CHANGE THIS: try 10, 40, or 80
exercise_plan = {
    row["feedback_trace_id"]: sampling_reason(row, BASELINE_RATE_PERCENT)
    for row in current_rows
}
assert all(exercise_plan[trace_id] == "must_keep_policy" for trace_id in policy_false_allow_ids)
print("Your-turn plan:", exercise_plan)
print("Your-turn correctness: PASS - routine sampling changed without weakening critical retention.")

**Reflection bridge:** Risk-aware sampling protects rare failures, but sampled rows can still be malformed or compare different populations. Validate the contract before calculating drift.

## 2 - Validate the Contract and Comparison Windows

Schema validation checks shape. Semantic checks verify that failure codes agree with the facts used to derive them. Both are required before a drift dashboard is allowed to speak.

```mermaid
flowchart LR
    A["JSONL rows"] --> B["Draft 2020-12 schema"]
    B --> C["Unique IDs + six rows/window"]
    C --> D["Re-derive failures"]
    D --> E{"Codes agree?"}
    E -->|no| X["Block report"]
    E -->|yes| F["Trusted analysis frame"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Trust failure codes because the row is schema-valid | Labels can disagree with measured fields |
| Right | Re-derive codes from source facts | Analysis tests its own inputs |
| Wrong | Compare unknown populations | Change has no interpretable baseline |
| Right | Pin window, release, count, and timestamps | Denominators stay auditable |

**Quick Health Check:** validate all rows, unique IDs, six rows per window, one release per window, and exact derived-code agreement.

In [ ]:
# -- Validate structural and semantic invariants before computing drift -------
from jsonschema import Draft202012Validator, FormatChecker

validator = Draft202012Validator(trace_schema, format_checker=FormatChecker())
validation_errors = [
    (row.get("feedback_trace_id"), list(error.path), error.message)
    for row in traces
    for error in validator.iter_errors(row)
]
LATENCY_SLO_MS = 140


def derive_failure_codes(row: dict) -> set[str]:
    codes = set()
    if not row["retrieval_hit"]:
        codes.add("retrieval_miss")
    if not row["quality_pass"]:
        codes.add("quality_failure")
    if row["latency_ms"] > LATENCY_SLO_MS:
        codes.add("latency_slo_breach")
    if row["expected_policy_outcome"] == "block" and row["actual_policy_outcome"] == "allow":
        codes.add("policy_false_allow")
    return codes


trace_ids = [row["feedback_trace_id"] for row in traces]
request_ids = [row["request_id"] for row in traces]
window_counts = Counter(row["window"] for row in traces)
window_releases = {
    window: {row["release_id"] for row in traces if row["window"] == window}
    for window in window_counts
}
semantic_mismatches = [
    row["feedback_trace_id"]
    for row in traces
    if set(row["failure_codes"]) != derive_failure_codes(row)
]
contract_health = {
    "schema_valid": not validation_errors,
    "trace_ids_unique": len(trace_ids) == len(set(trace_ids)),
    "request_ids_unique": len(request_ids) == len(set(request_ids)),
    "six_rows_per_window": window_counts == {"baseline": 6, "current": 6},
    "one_release_per_window": all(len(releases) == 1 for releases in window_releases.values()),
    "failure_codes_reconcile": not semantic_mismatches,
}
for name, passed in contract_health.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(contract_health.values())
print("Window releases:", window_releases)

**Reflection bridge:** Valid windows make arithmetic trustworthy, but one aggregate still cannot identify the component owner. The same rows now need separate traffic, data, retrieval, quality, operations, cost, and policy lenses.

## 3 - Seven Drift Lenses, Seven Different Owners

Traffic drift asks who arrives. Data drift asks what arrives within that traffic. Retrieval drift asks whether evidence arrives. Quality drift asks whether the task contract passes. Latency and cost drift ask whether the route remains operable. Policy drift asks whether deterministic authority matches expectation. None alone is `model drift`.

| Lens | First question | Likely owner |
|---|---|---|
| Traffic | Did the slice/route mix change? | Product and routing |
| Data | Did input structure/category novelty change? | Data and instrumentation |
| Retrieval | Did authorized evidence reach the prompt? | Index and ingestion |
| Quality | Did the final task contract pass? | Evaluation and application |
| Latency | Did user-visible time or SLO breach change? | Gateway and serving |
| Cost | Did spend per observed request change? | Gateway, serving, finance |
| Policy | Did authority match expectation? | Security and policy |

```mermaid
flowchart TD
    A["Window comparison"] --> T["Traffic mix"]
    A --> D["Input novelty"]
    A --> R["Retrieval hit rate"]
    A --> Q["Quality pass rate"]
    A --> L["Latency mean, p95, SLO"]
    A --> C["Mean cost"]
    A --> P["Policy correctness"]
    T --> O["Diagnose owner before tuning"]
    D --> O
    R --> O
    Q --> O
    L --> O
    C --> O
    P --> O
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If all three quality failures are retrieval misses, should Riverside first rewrite the prompt, update retrieval, or fine-tune?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Call every change model drift | The model owns unrelated failures |
| Right | Name the observable layer and next test | Intervention follows evidence |
| Wrong | Average policy and quality together | A false allow disappears inside a mean |
| Right | Keep critical policy outcomes as hard gates | Authority remains visible |

**Quick Health Check:** every signal states window, population or units, direction, and diagnostic owner.

In [ ]:
# -- Compute deterministic traffic, data, retrieval, quality, ops, and policy drift
baseline_rows = [row for row in traces if row["window"] == "baseline"]
current_rows = [row for row in traces if row["window"] == "current"]


def rate(rows, predicate):
    numerator = sum(bool(predicate(row)) for row in rows)
    return numerator, len(rows), numerator / len(rows)


def arithmetic_mean(rows, field):
    return sum(row[field] for row in rows) / len(rows)


def nearest_rank(values, quantile):
    ordered = sorted(values)
    return ordered[max(1, math.ceil(quantile * len(ordered))) - 1]


baseline_summaries = {row["query_summary"] for row in baseline_rows}
novel_current = [row for row in current_rows if row["query_summary"] not in baseline_summaries]
drift_metrics = {
    "traffic.security_share": (
        rate(baseline_rows, lambda row: row["slice"] == "security"),
        rate(current_rows, lambda row: row["slice"] == "security"),
    ),
    "data.novel_summary_rate": ((0, 6, 0.0), (len(novel_current), 6, len(novel_current) / 6)),
    "retrieval.hit_rate": (
        rate(baseline_rows, lambda row: row["retrieval_hit"]),
        rate(current_rows, lambda row: row["retrieval_hit"]),
    ),
    "quality.pass_rate": (
        rate(baseline_rows, lambda row: row["quality_pass"]),
        rate(current_rows, lambda row: row["quality_pass"]),
    ),
    "latency.slo_breach_rate": (
        rate(baseline_rows, lambda row: row["latency_ms"] > LATENCY_SLO_MS),
        rate(current_rows, lambda row: row["latency_ms"] > LATENCY_SLO_MS),
    ),
    "policy.correct_rate": (
        rate(baseline_rows, lambda row: row["expected_policy_outcome"] == row["actual_policy_outcome"]),
        rate(current_rows, lambda row: row["expected_policy_outcome"] == row["actual_policy_outcome"]),
    ),
}
for name, (baseline, current) in drift_metrics.items():
    print(
        f"{name:30s} B={baseline[0]}/{baseline[1]} ({baseline[2]:.1%}) "
        f"C={current[0]}/{current[1]} ({current[2]:.1%}) "
        f"change={100 * (current[2] - baseline[2]):+.1f} pp"
    )

baseline_latency = arithmetic_mean(baseline_rows, "latency_ms")
current_latency = arithmetic_mean(current_rows, "latency_ms")
baseline_cost = arithmetic_mean(baseline_rows, "cost_microusd")
current_cost = arithmetic_mean(current_rows, "cost_microusd")
baseline_p95 = nearest_rank([row["latency_ms"] for row in baseline_rows], 0.95)
current_p95 = nearest_rank([row["latency_ms"] for row in current_rows], 0.95)
print(f"Latency mean: {baseline_latency:.0f} -> {current_latency:.0f} ms; p95: {baseline_p95} -> {current_p95} ms")
print(f"Cost mean: {baseline_cost:.0f} -> {current_cost:.0f} micro-USD")
print("Novel summaries:", [row["query_summary"] for row in novel_current])

assert drift_metrics["traffic.security_share"][0][2] == 1 / 6
assert drift_metrics["traffic.security_share"][1][2] == 3 / 6
assert len(novel_current) == 2
assert drift_metrics["retrieval.hit_rate"][0][2] == 5 / 6
assert drift_metrics["retrieval.hit_rate"][1][2] == 3 / 6
assert drift_metrics["quality.pass_rate"][0][2] == 5 / 6
assert drift_metrics["quality.pass_rate"][1][2] == 3 / 6
assert drift_metrics["latency.slo_breach_rate"][0][0] == 0
assert drift_metrics["latency.slo_breach_rate"][1][0] == 3
assert drift_metrics["policy.correct_rate"][0][2] == 1
assert drift_metrics["policy.correct_rate"][1][2] == 5 / 6
assert (baseline_latency, current_latency, baseline_cost, current_cost) == (100, 150, 1000, 1500)
assert (baseline_p95, current_p95) == (110, 200)
print("PASS: all deterministic drift expectations reproduced.")
print("Prediction resolved: retrieval/index work comes before prompt rewriting or fine-tuning because every quality failure is a retrieval miss.")

**Reflection bridge:** Seven lenses localize the changes, but six rows per window can make large percentage-point deltas look more certain than they are. Quantify uncertainty without averaging away critical events.

## 4 - Uncertainty: Six Rows Are a Warning Label

For a binary rate, the Wilson interval behaves better than a naive normal interval near 0% and 100%:

$$center = \frac{p + z^2/(2n)}{1 + z^2/n}, \qquad half = \frac{z}{1 + z^2/n}\sqrt{\frac{p(1-p)}{n}+\frac{z^2}{4n^2}}$$

The interval is `center +/- half`. It quantifies binomial sampling uncertainty under assumptions this fixture does not fully satisfy. It does not correct biased sampling, repeated monitoring, release confounding, evaluator error, or traffic shift.

```mermaid
flowchart LR
    A["Point estimate"] --> B["Wilson interval"]
    B --> C{"Critical event?"}
    C -->|yes| D["Escalate regardless of mean"]
    C -->|no| E{"Representative sample?"]
    E -->|no| X["Review / collect more"]
    E -->|yes| F["Apply predeclared gate"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Will three successes in six observations produce a narrow or wide 95% interval?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat a large percentage change as significant | Tiny denominators are unstable |
| Right | Report effect, interval, sample size, and sampling policy | Decision strength stays visible |
| Wrong | Wait for significance before containing a false allow | Safety is not an average-quality question |
| Right | Escalate critical events; analyze prevalence separately | Containment does not wait |

**Quick Health Check:** every proportion has sample size and interval; critical policy events bypass ordinary statistical gating.

**Your turn:** Change the practical-review threshold in the next cell. Predict which ordinary signals leave the review queue; the policy false allow must remain independently escalated.

In [ ]:
# -- Add Wilson intervals without manufacturing a significance claim ---------
def wilson_interval(successes: int, total: int, z: float = 1.96):
    if total <= 0 or not 0 <= successes <= total:
        raise ValueError("Invalid binomial count.")
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    half = (z / denominator) * math.sqrt(
        proportion * (1 - proportion) / total + z * z / (4 * total * total)
    )
    return max(0.0, center - half), min(1.0, center + half)


for name, (baseline, current) in drift_metrics.items():
    baseline_ci = wilson_interval(baseline[0], baseline[1])
    current_ci = wilson_interval(current[0], current[1])
    print(
        f"{name:30s} B={baseline[2]:.1%} CI=[{baseline_ci[0]:.1%}, {baseline_ci[1]:.1%}] "
        f"C={current[2]:.1%} CI=[{current_ci[0]:.1%}, {current_ci[1]:.1%}] n=6"
    )
assert wilson_interval(3, 6)[1] - wilson_interval(3, 6)[0] > 0.5
assert policy_false_allow_ids
print("PASS: intervals expose low precision; the false allow remains an immediate gate.")
print("Prediction resolved: three of six yields a wide interval, not release-grade certainty.")

# Your turn: change triage volume, not statistical power.
MIN_ABSOLUTE_CHANGE_PP = 20.0  # CHANGE THIS: try 10, 20, or 40
review_signals = [
    (name, 100 * (current[2] - baseline[2]))
    for name, (baseline, current) in drift_metrics.items()
    if abs(100 * (current[2] - baseline[2])) >= MIN_ABSOLUTE_CHANGE_PP
]
print("Practical review signals:", review_signals)
print("Critical policy review required independently:", bool(policy_false_allow_ids))
print("Your-turn correctness: PASS - the threshold changes triage volume, not critical-event escalation.")

**Reflection bridge:** Uncertainty limits confidence in prevalence, but it does not tell reviewers which overlapping incidents deserve attention first. Preserve multi-label membership and rank by risk.

## 5 - Cluster and Review Failures Without Erasing Overlap

A request can belong to several groups. `fb-010` is a retrieval miss, quality failure, latency breach, and policy false allow. Forcing one cluster discards evidence. The fixture builds an inverted index by failure code and an exact-signature view. These are interpretable buckets, not a claim that unsupervised learning discovered a production taxonomy.

```mermaid
flowchart LR
    A["Failed traces"] --> B["Multi-label code index"]
    A --> C["Exact signatures"]
    B --> D["Risk-aware review queue"]
    C --> D
    D --> E["Human label + rationale"]
    E --> F["Reviewed regression case"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which trace should review first: a common retrieval-only failure or the trace combining retrieval, quality, latency, and policy failure?

A reviewer sees the privacy-safe summary, release, slice, measured facts, expected policy outcome, and existing codes. The reviewer supplies a label and rationale through an audited interface; user thumbs-down only nominates the case.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Cluster embeddings over raw prompts by default | Sensitive text and operational causes are mixed |
| Right | Start with bounded features; approve content analysis separately | Buckets stay interpretable |
| Wrong | Review only the largest cluster | Rare policy failures never become regressions |
| Right | Prioritize severity, then cluster and slice coverage | Budget reflects risk |

**Quick Health Check:** membership is multi-label, every failure is covered, critical events rank first, and thumbs-down is not treated as a label.

In [ ]:
# -- Build deterministic multi-label clusters and verify reviewed labels ------
failed_current = [row for row in current_rows if row["failure_codes"]]
clusters = defaultdict(list)
signatures = defaultdict(list)
for row in failed_current:
    for code in row["failure_codes"]:
        clusters[code].append(row["feedback_trace_id"])
    signatures[tuple(sorted(row["failure_codes"]))].append(row["feedback_trace_id"])


def review_priority(row):
    return (
        0 if "policy_false_allow" in row["failure_codes"] else 1,
        -len(row["failure_codes"]),
        row["feedback_trace_id"],
    )


review_queue = sorted(failed_current, key=review_priority)
reviewed_rows = [row for row in current_rows if row["review_status"] == "reviewed"]
expected_clusters = {
    "retrieval_miss": ["fb-009", "fb-010", "fb-012"],
    "quality_failure": ["fb-009", "fb-010", "fb-012"],
    "latency_slo_breach": ["fb-010", "fb-011", "fb-012"],
    "policy_false_allow": ["fb-010"],
}
for code, ids in sorted(clusters.items()):
    print(f"{code:24s} {ids}")
print("Signatures:", dict(signatures))
print("Review order:", [row["feedback_trace_id"] for row in review_queue])
review_health = {
    "expected_clusters": dict(clusters) == expected_clusters,
    "critical_first": review_queue[0]["feedback_trace_id"] == "fb-010",
    "all_failures_covered": (
        {row["feedback_trace_id"] for row in failed_current}
        == set().union(*map(set, clusters.values()))
    ),
    "three_reviewed": len(reviewed_rows) == 3,
    "labels_present": all(row["review_label"] for row in reviewed_rows),
    "downvote_not_label": all(
        row["review_label"] != row["user_feedback"] for row in reviewed_rows
    ),
}
for name, passed in review_health.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(review_health.values())
print("Prediction resolved: fb-010 ranks first because it combines policy, retrieval, quality, and latency failures.")

**Reflection bridge:** A reviewed queue creates labels, but labels in memory are not reusable release gates. Bind each case to its source release, evaluator contract, canonical bytes, and digest.

## 6 - Build a Versioned Evaluation Candidate

A production trace is not automatically a golden test. Promotion requires a reviewed label, expected outcome, reproducible representation, source release lineage, evaluator contract, and versioned dataset identity. This fixture has no raw production text, so the candidate preserves only the synthetic query summary.

```mermaid
flowchart LR
    A["Reviewed trace"] --> B["Privacy-safe candidate"]
    B --> C["Canonical serialization"]
    C --> D["SHA-256 digest"]
    D --> E["Versioned manifest"]
    E --> F["Offline gate after fix"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which proves origin: row order, query category, or originating trace plus release lineage?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Copy raw production text into git | Evaluation growth leaks content |
| Right | Use approved synthetic/replayed representation with lineage | Regression value survives safely |
| Wrong | Append to an unversioned golden CSV | A release cannot reproduce its gate |
| Right | Pin version, source digest, evaluator, and case digest | The gate is auditable |

**Quick Health Check:** only reviewed traces become cases, IDs map one-to-one, lineage remains, forbidden fields are absent, and canonical bytes hash stably.

In [ ]:
# -- Convert reviewed feedback into a privacy-safe versioned candidate --------
def required_checks(label):
    if label == "policy_failure":
        return ["policy_outcome_matches_expected", "retrieval_hit", "quality_pass"]
    if label == "retrieval_failure":
        return ["retrieval_hit", "quality_pass"]
    return ["quality_pass"]


def candidate_case(row):
    if row["review_status"] != "reviewed":
        raise ValueError("Only reviewed feedback may enter.")
    return {
        "schema_version": "ai-eng.eval-candidate-case.v1",
        "candidate_eval_case_id": row["candidate_eval_case_id"],
        "query_summary": row["query_summary"],
        "slice": row["slice"],
        "expected_policy_outcome": row["expected_policy_outcome"],
        "review_label": row["review_label"],
        "required_checks": required_checks(row["review_label"]),
        "source": {
            "feedback_trace_id": row["feedback_trace_id"],
            "request_id": row["request_id"],
            "release_id": row["release_id"],
            "observed_at_utc": row["observed_at_utc"],
            "failure_codes": sorted(row["failure_codes"]),
        },
    }


def canonical_bytes(value):
    return json.dumps(
        value, ensure_ascii=True, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")


candidate_cases = [
    candidate_case(row)
    for row in sorted(reviewed_rows, key=lambda row: row["candidate_eval_case_id"])
]
cases_sha256 = sha256(canonical_bytes(candidate_cases)).hexdigest()
candidate_manifest = {
    "schema_version": "ai-eng.eval-candidate-set.v1",
    "dataset_id": "riverside-production-feedback-candidates",
    "version": "2026-02-20.1",
    "evaluator_contract_version": "feedback-drift-evaluator.v1",
    "source_fixture_sha256": fixture_sha256,
    "source_release_ids": sorted(
        {case["source"]["release_id"] for case in candidate_cases}
    ),
    "case_count": len(candidate_cases),
    "cases_sha256": cases_sha256,
    "cases": candidate_cases,
}
serialized = canonical_bytes(candidate_manifest).decode("utf-8")
candidate_health = {
    "three_cases": len(candidate_cases) == 3,
    "expected_ids": [case["candidate_eval_case_id"] for case in candidate_cases]
    == ["evalcand-001", "evalcand-002", "evalcand-003"],
    "one_to_one_source": len(
        {case["source"]["feedback_trace_id"] for case in candidate_cases}
    ) == 3,
    "release_lineage": candidate_manifest["source_release_ids"] == ["rel-riv-002"],
    "no_raw_content_keys": not any(
        key in serialized
        for key in ("raw_prompt", "completion", "manuscript_text", "user_id", "tenant_id")
    ),
    "stable_digest": sha256(canonical_bytes(candidate_cases)).hexdigest()
    == cases_sha256,
}
print(json.dumps(candidate_manifest, indent=2))
for name, passed in candidate_health.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(candidate_health.values())

WRITE_CANDIDATE_ARTIFACTS = False  # CHANGE THIS only after repository review
if WRITE_CANDIDATE_ARTIFACTS:
    artifact_dir = CHAPTER_DIR / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    artifact_path = artifact_dir / (
        f"{candidate_manifest['dataset_id']}-{candidate_manifest['version']}.json"
    )
    artifact_path.write_text(
        json.dumps(candidate_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print(f"Wrote reviewed candidate: {artifact_path}")
else:
    print("Artifact write disabled; candidate exists only in notebook memory.")
print("Prediction resolved: originating trace plus release lineage proves origin; row order and category do not.")

**Reflection bridge:** A versioned candidate makes the failure reproducible, but it does not choose the owner. Route each reviewed symptom through the evidence chain before changing prompts or weights.

## 7 - Choose the Intervention the Evidence Supports

| Action | Supporting evidence | Fixture verdict |
|---|---|---|
| Prompt change | Correct evidence arrives and policy is correct, but output contract fails | Not isolated |
| Retrieval/index update | Quality failures coincide with missing or stale evidence | Supported: 3/3 |
| Guardrail change | Expected and actual policy outcomes disagree | Mandatory: one false allow |
| Fine-tune | Retrieval/policy are healthy and persistent learned behavior fails held-out review | Not supported first |
| No action | No critical event or representative follow-up is healthy | Not supported now |

```mermaid
flowchart TD
    A["Reviewed failure"] --> B{"Evidence retrieved?"}
    B -->|no| R["Retrieval/index update"]
    B -->|yes| C{"Policy correct?"}
    C -->|no| G["Guardrail change"]
    C -->|yes| D{"Output behavior failed?"}
    D -->|prompt-owned| P["Prompt/application change"]
    D -->|persistent learned gap| F["Fine-tune with held-out gate"]
    D -->|no material signal| N["No action"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style N fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Will the record recommend one action or several parallel actions?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Pick one cause for a multi-layer incident | Policy and retrieval defects can coexist |
| Right | Separate containment, correction, and investigation | Priority and ownership stay clear |
| Wrong | Fine-tune current policy facts into weights | Facts stale; authority remains unenforced |
| Right | Retrieve facts; enforce authority outside the model | Knowledge and policy stay inspectable |

**Quick Health Check:** selected and rejected actions each cite evidence, and policy repair never depends on a quality average.

In [ ]:
# -- Produce a deterministic multi-action decision with explicit rejections --
quality_failures = [row for row in current_rows if not row["quality_pass"]]
retrieval_misses = [row for row in current_rows if not row["retrieval_hit"]]
quality_with_hit = [row for row in quality_failures if row["retrieval_hit"]]
policy_false_allows = [
    row
    for row in current_rows
    if row["expected_policy_outcome"] == "block"
    and row["actual_policy_outcome"] == "allow"
]
all_quality_failures_are_retrieval_misses = (
    {row["feedback_trace_id"] for row in quality_failures}
    <= {row["feedback_trace_id"] for row in retrieval_misses}
)
selected_actions = []
if quality_failures and all_quality_failures_are_retrieval_misses:
    selected_actions.append(
        {
            "action": "retrieval/index update",
            "priority": "immediate correction",
            "evidence": "3/3 quality failures are retrieval misses",
            "next_test": "rerun evalcand-001..003 on candidate index",
        }
    )
if policy_false_allows:
    selected_actions.append(
        {
            "action": "guardrail change",
            "priority": "immediate containment",
            "evidence": "fb-010 is a false allow",
            "next_test": "fail-closed test independent of generation",
        }
    )
if current_latency > baseline_latency or current_cost > baseline_cost:
    selected_actions.append(
        {
            "action": "latency/cost investigation",
            "priority": "parallel diagnosis",
            "evidence": "mean latency and cost increased 50%",
            "next_test": "stage trace by security route",
        }
    )
rejected_actions = [
    {
        "action": "prompt change",
        "reason": f"{len(quality_with_hit)} quality failures had correct retrieval",
    },
    {
        "action": "fine-tune",
        "reason": "no persistent behavior gap remains after healthy retrieval and policy",
    },
    {
        "action": "no action",
        "reason": "material drift and one false allow require response",
    },
]
decision_record = {
    "decision_id": "feedback-decision-rel-riv-002-001",
    "source_release_id": "rel-riv-002",
    "candidate_dataset": f"{candidate_manifest['dataset_id']}@{candidate_manifest['version']}",
    "selected_actions": selected_actions,
    "rejected_actions": rejected_actions,
    "status": "HOLD_RELEASE_AND_REMEDIATE",
}
print(json.dumps(decision_record, indent=2))
selected_names = {item["action"] for item in selected_actions}
rejected_names = {item["action"] for item in rejected_actions}
assert {"retrieval/index update", "guardrail change"} <= selected_names
assert {"prompt change", "fine-tune", "no action"} == rejected_names
print("PASS: retrieval and guardrail changes lead; operations investigates in parallel.")

**Reflection bridge:** The evidence supports several parallel actions rather than one convenient cause. Those actions still need offline gates, bounded rollout, a follow-up window, and rollback.

## 8 - Operate the Loop, Not the Notebook

A candidate and action record are intermediate artifacts. The loop closes only when Riverside versions the index and guardrail changes, runs the candidate before release, observes a representative follow-up window, and advances or rolls back from retained evidence.

```mermaid
flowchart LR
    A["Privacy-safe window"] --> B["Drift report"]
    B --> C["Cluster + review"]
    C --> D["Versioned candidate"]
    D --> E["Index + guardrail candidate"]
    E --> F{"Offline gates pass?"}
    F -->|no| X["Hold / repair"]
    F -->|yes| G["Shadow then canary"]
    G --> H{"Follow-up healthy?"]
    H -->|no| X
    H -->|yes| A
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Production health checks

1. Sampling coverage by release and slice remains within the approved plan.
2. Policy and safety events are retained at 100% in privacy-safe metadata.
3. Schema, instrumentation, evaluator, release, prompt, and index versions are present.
4. Every dashboard shows counts, denominators, windows, and freshness.
5. Retrieval and generation remain separate; use a gold-context ablation when ownership is unclear.
6. Reviewer agreement and backlog age are monitored; user feedback is not ground truth.
7. Candidate cases have provenance, digests, retention, and deletion rules.
8. Critical policy gates cannot be averaged or sampled away.
9. Latency and cost compare the same route and workload before claiming regression.
10. Every action has an owner, next test, observation window, and rollback target.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Add every complaint permanently | Noise and stale policy overwhelm signal |
| Right | Review, deduplicate, version, retire, and audit cases | Benchmark stays intentional |
| Wrong | Declare success after offline cases pass | Production distribution remains untested |
| Right | Require bounded rollout and representative follow-up | Improvement is checked in context |

**Quick Health Check:** contracts, drift, review, candidate lineage, action evidence, owners, observation window, and rollback target must all be present.

In [ ]:
# -- Consolidate chapter evidence into one operating health report -----------
operating_health = {
    **{f"contract.{name}": passed for name, passed in contract_health.items()},
    **{f"sampling.{name}": passed for name, passed in sampling_health.items()},
    **{f"review.{name}": passed for name, passed in review_health.items()},
    **{f"candidate.{name}": passed for name, passed in candidate_health.items()},
    "diagnosis.quality_subset_retrieval": (
        {row["feedback_trace_id"] for row in quality_failures}
        <= {row["feedback_trace_id"] for row in retrieval_misses}
    ),
    "diagnosis.false_allow_visible": bool(policy_false_allows),
    "decision.correct_actions": (
        {"retrieval/index update", "guardrail change"} <= selected_names
    ),
    "decision.fine_tune_not_first": "fine-tune" in rejected_names,
}
for name, passed in operating_health.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(operating_health.values())

follow_up_plan = {
    "baseline_release_id": "rel-riv-002",
    "candidate_changes": [
        "versioned policy index update",
        "fail-closed guardrail update",
    ],
    "offline_dataset": f"{candidate_manifest['dataset_id']}@{candidate_manifest['version']}",
    "must_pass": [
        "3/3 candidate retrieval checks",
        "0 policy false allows",
        "no critical slice regression",
    ],
    "parallel_investigation": "stage-level security-route latency and cost attribution",
    "rollout": "shadow, then bounded canary after offline gates",
    "rollback_target": "rel-riv-002 plus prior accepted index and policy",
    "no_action_rule": "only after a representative healthy follow-up window",
}
print(json.dumps(follow_up_plan, indent=2))

**Reflection bridge:** An operating loop can retain trustworthy evidence while still overclaiming what twelve synthetic traces establish. Close by separating built mechanics from production validation.

## 9 - Coverage and Honest Limits

```mermaid
flowchart LR
    A["12 synthetic traces"] --> B["Arithmetic + lineage + decision order"]
    B --> C["Not production representativeness"]
    C --> D["Collect approved follow-up evidence"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Three-tier coverage ledger

| Tier | Techniques | Reason |
|---|---|---|
| Built and measured | Data minimization check; deterministic sampling; must-keep safety strata; schema/semantic validation; traffic/data/retrieval/quality/latency/cost/policy drift; nearest-rank p95; Wilson intervals; multi-label clusters; review queue; versioned candidate; canonical SHA-256; intervention matrix; operating health | Fixture proves these mechanics deterministically |
| Explained and illustrated | Head/tail sampling; stratification; gold-context ablation; reviewer agreement; EWMA; shadow/canary follow-up; case retirement | Needs more history, people, or runtime |
| Named with a reason | PSI/KS; embedding drift; topic/density clustering; learned anomalies; causal impact; live OTel backend; reviewer UI | Twelve categorical traces cannot validate these responsibly |

If a technique is named elsewhere but absent here, treat that as a coverage bug.

### Completed roadmap

| Step | Expected fixture evidence |
|---|---|
| Privacy-safe sampling | Forbidden fields absent; policy false allow must-kept |
| Contract health | 12 valid rows; six per window; failure codes agree |
| Drift diagnosis | Traffic +33.3 pp; retrieval/quality -33.3 pp; latency/cost +50%; policy -16.7 pp |
| Uncertainty | Wilson intervals and `n=6` printed |
| Cluster/review | Four overlapping code groups; false allow first; three reviewed cases |
| Eval candidate | `evalcand-001..003`, release lineage, canonical digest |
| Intervention | Index + guardrail; operations investigation; prompt/fine-tune/no-action rejected first |
| Operating loop | Offline rerun, shadow/canary, rollback target, healthy follow-up rule |

### Key takeaways

- Production feedback is a governed data product, not permission to log everything.
- Traffic drift changes who arrives; data drift changes what arrives; neither proves model drift.
- Retrieval, generation quality, policy, latency, and cost need separate gates.
- Rare safety events are must-keep evidence even when prevalence is uncertain.
- User feedback nominates review; it is not ground truth.
- A reviewed case needs source lineage, expected behavior, version, evaluator, and digest.
- Fine-tune only after retrieval and policy are healthy and a learned-behavior gap remains.
- No action is earned by representative follow-up evidence, not dashboard silence.

### What this notebook cannot prove

The fixture cannot establish production representativeness, semantic data drift, causal impact, alert calibration, reviewer agreement, privacy compliance, deployed index quality, deployed policy correctness, provider billing, concurrency capacity, or cloud telemetry delivery. Six observations per window are mechanism evidence only. Production claims require approved sampling, representative traffic, versioned release/index/policy state, calibrated evaluation, human review, bounded rollout, incident response, and retained follow-up evidence.